# Playing the Railroad environment against the bosses

Everything you need to write an agent, watch it play, and benchmark it against the whole
boss ladder. No training here and no PyTorch required.

Run top to bottom. The only prerequisite is `pip install -r requirements.txt`.


In [1]:
%matplotlib inline
import sys, time
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '.')
from railroad_env import RailroadGymEnv, BOSS_TIERS, make_opponent
from railroad_env.game_state import GameState
from encoding import encode_state, build_place_mask

print('bosses, weakest to strongest:', BOSS_TIERS)
print('observation channels:', GameState.NUM_CHANNELS)


bosses, weakest to strongest: ('level1', 'level1Pro', 'level2', 'level2Pro', 'level2ProMax')
observation channels: 28


## 1. One game, start to finish

You are always player 0; the boss is player 1. `env.step` takes every action for your whole
turn at once, so spend the full paint budget in one call.


In [2]:
env = RailroadGymEnv(max_turns=100, opponent_strategy='level2', seed=0)
obs, info = env.reset()
gs = env.game_state

print(f'map {gs.height}x{gs.width}, {len(gs.grid.towns)} towns, {len(gs.zones)} regions')
print(f'observation {obs.shape}  (height, width, channels)')
print(f'paint per turn {gs.paint_points[0]}, disruption points {gs.disruption_points[0]}')

for town in gs.grid.towns[:4]:
    wants = [t.id for t in town.desired_connections]
    print(f'  town {town.id} at {town.coord} wants {wants}')


map 18x27, 9 towns, 54 regions
observation (18, 27, 28)  (height, width, channels)
paint per turn 3, disruption points 1
  town 0 at (12, 2) wants [5, 6, 8]
  town 1 at (3, 3) wants [0]
  town 2 at (15, 4) wants [1, 7, 8]
  town 3 at (9, 5) wants [1, 2, 4]


## 2. A baseline agent

This one is deliberately simple, so it is easy to see what you would replace. Each turn it
spends its paint on the cheapest legal cells that sit on a route guide, preferring cells
next to track it already owns. It never disrupts.

Two things any agent must respect: check `_is_placeable` before placing, and track your own
budget as you spend it, because the referee silently drops actions you cannot afford.


In [3]:
ADJ = ((0, -1), (1, 0), (0, 1), (-1, 0))

def greedy_agent(gs):
    """Cheapest legal cells that lie on a route guide, touching our own network first."""
    obs = gs.get_observation()
    guide = obs[:, :, 10:22].max(axis=2)      # >0 means 'some town's route runs here'
    budget = gs.paint_points[0]

    candidates = []
    for y in range(gs.height):
        for x in range(gs.width):
            if not gs._is_placeable(x, y) or guide[y, x] <= 0:
                continue
            touching = sum(
                1 for dx, dy in ADJ
                if (t := gs.grid.get(x + dx, y + dy)) is not None
                and (t.track == 0 or t.is_town())
            )
            candidates.append((-touching, gs.get_track_cost(x, y), x, y))
    candidates.sort()

    actions = []
    for _, cost, x, y in candidates:
        if cost <= budget:
            actions.append(('PLACE', x, y))
            budget -= cost
    return actions or [('WAIT',)]


def play(agent, boss, seed=0, max_turns=100):
    env = RailroadGymEnv(max_turns=max_turns, opponent_strategy=boss, seed=seed)
    env.reset()
    gs = env.game_state
    while not gs.is_done():
        env.step({'actions': agent(gs)})
    return gs.scores[0], gs.scores[1]

mine, theirs = play(greedy_agent, 'level2', seed=0)
print(f'vs level2 -> you {mine}, boss {theirs}, margin {mine - theirs:+d}')


vs level2 -> you 18131, boss 7114, margin +11017


## 3. The whole boss ladder

Always benchmark on a **fixed set of seeds**. Maps are generated per seed, so reusing the
same ones is what makes two runs comparable; change them and you are measuring map luck.


In [4]:
SEEDS = list(range(12))          # raise for tighter numbers, these are noisy

print(f"{'boss':>14} {'you':>7} {'boss':>7} {'margin':>8} {'win':>6}  {'time':>6}")
print('-' * 56)
for boss in BOSS_TIERS:
    t0 = time.perf_counter()
    rows = [play(greedy_agent, boss, seed=s) for s in SEEDS]
    a = np.mean([r[0] for r in rows]); b = np.mean([r[1] for r in rows])
    win = np.mean([r[0] > r[1] for r in rows])
    print(f'{boss:>14} {a:>7.0f} {b:>7.0f} {a-b:>+8.0f} {win:>6.2f}  {time.perf_counter()-t0:>5.1f}s')


          boss     you    boss   margin    win    time
--------------------------------------------------------


        level1   17542       0   +17542   1.00    1.7s


     level1Pro   16537    1132   +15406   1.00    2.3s


        level2   14917    6435    +8482   1.00    2.6s


     level2Pro   14944    7264    +7679   1.00    8.7s


  level2ProMax    8732   11190    -2458   0.17   31.2s


Expect a cliff at `level2ProMax`. It finishes building around turn 25 and then spends the
rest of the game inking your regions, one every four turns, while its own connections keep
paying. Regions containing a town can never be inked and are the only safe ground.


## 4. What the agent actually sees

All 28 channels of one mid-game position. This is the most useful debugging tool here: if a
channel looks empty, constant, or wrong, your agent cannot use it.


In [5]:
CHANNEL_NAMES = [
    'terrain: plains', 'terrain: river', 'terrain: mountain',
    'enemy track', 'own track',
    'enemy count /10 (region)', 'own count /10 (region)', 'region size (norm)',
    'enemy active /10', 'own active /10',
] + [f'town {i} route guide' for i in range(GameState.NUM_TOWN_CHANNELS)] + [
    'instability /4', 'region inked', 'active connection',
    'towns', 'score gap /10k (flat)', 'turn /100 (flat)',
]
assert len(CHANNEL_NAMES) == GameState.NUM_CHANNELS

env = RailroadGymEnv(max_turns=100, opponent_strategy='level2ProMax', seed=3)
env.reset(); gs = env.game_state
for _ in range(40):
    if gs.is_done():
        break
    env.step({'actions': greedy_agent(gs)})

obs = gs.get_observation()
print(f'turn {gs.turn}, scores {gs.scores}')

cols = 6
rows = int(np.ceil(GameState.NUM_CHANNELS / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 2.0 * rows))
for i, ax in enumerate(axes.ravel()):
    if i >= GameState.NUM_CHANNELS:
        ax.axis('off'); continue
    plane = obs[:, :, i]
    # Route guides are signed (-1 town, +1 path), so centre those on zero to keep the sign
    # readable; the rest are non-negative and read better on a sequential map.
    signed = plane.min() < 0
    ax.imshow(plane, cmap='coolwarm' if signed else 'viridis',
              vmin=-1 if signed else 0, vmax=1 if signed else max(plane.max(), 1e-6))
    ax.set_title(f'{i}: {CHANNEL_NAMES[i]}', fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f'{GameState.NUM_CHANNELS} observation channels, turn {gs.turn}', fontsize=13)
fig.tight_layout()
plt.show()


turn 40, scores [1599, 1975]


/tmp/ipykernel_571771/2412300069.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Reading it: channels 26 and 27 are flat by design, since a convolution is local and cannot
otherwise see a board-wide scalar. The route guides are signed, `-1` at the town owning that
channel, `-0.5` at a town it wants to reach, `+1` on the cells between. Channels above the
town count stay empty on maps with fewer towns.


## 5. The network-ready tensor

Maps vary in size, so `encode_state` centres the board on a fixed 20x30 canvas and marks the
padding as inked. That gives a constant shape you can batch. Pair it with the action masks:
most cells are illegal on any given turn, and an unmasked policy burns its capacity
rediscovering that.


In [6]:
state = encode_state(gs)
mask = build_place_mask(gs, paint_budget=gs.paint_points[0])
print(f'state {state.shape} channels-first, mask {mask.shape}, {int(mask.sum())} legal cells')

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].imshow(state[4], cmap='viridis'); ax[0].set_title('ch4 own track, padded canvas')
ax[1].imshow(state[23], cmap='viridis'); ax[1].set_title('ch23 inked (padding marked inked)')
ax[2].imshow(mask[0], cmap='gray');      ax[2].set_title('legal placements right now')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()


state (28, 20, 30) channels-first, mask (1, 20, 30), 164 legal cells


/tmp/ipykernel_571771/1765265681.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Where to go next

Swap `greedy_agent` for anything with the same shape: take a `GameState`, return a list of
actions. To train a network, use `encode_state` for the input and the masks for the output.

Three things worth knowing before you tune anything:

- **The scoring path is the shortest in *cells*, not in build cost.** A detour around a
  mountain is cheaper to build, but only the fewest-cell path pays.
- **Inking destroys both players' track in a region.** Concentrating your network makes it a
  target; roughly 14% of the board sits in town regions and can never be inked.
- **Fix your seeds when benchmarking.** Otherwise you are measuring the maps.
